In [1]:
import pandas as pd
import numpy as np
import re,string

In [2]:
pip install deep-translator

In [3]:
from deep_translator import GoogleTranslator

def translate_safe(text):
    if len(str(text)) > 500:
        return text

    return GoogleTranslator(source="auto", target="en").translate(text)

In [4]:
train_df=pd.read_csv("/content/train.csv")

In [5]:
test_df=pd.read_csv("/content/test.csv")

In [6]:
test_df.dropna(inplace=True)

In [7]:
train_df.isna().sum()

,0
id,0
comment_text,0
toxic,0
severe_toxic,0
obscene,0
threat,0
insult,0
identity_hate,0


In [8]:
train_df.dropna(inplace=True)

In [9]:
train_df.shape

(159571, 8)

In [10]:
train_df.drop(columns=["id"],inplace=True)

In [11]:
import spacy

In [12]:
nlp = spacy.load("en_core_web_sm")

In [13]:
!pip install emoji

In [14]:
import emoji

def convert_emoji(text):
    return emoji.demojize(text, delimiters=(" ", " "))

In [15]:
def has_emoji(text):
    if not isinstance(text, str):
        return 0
    return int(any(char in emoji.EMOJI_DATA for char in text))

In [16]:
text = "you are stupid 😡"
print(convert_emoji(text))

you are stupid  enraged_face 


In [17]:
def normalize_leetspeak(text):
    patterns = {
        r'[4@]': 'a',
        r'[8]': 'b',
        r'[\(\{\[]': 'c',
        r'[3]': 'e',
        r'[6|9]': 'g',
        r'[1!|]': 'i',
        r'[0]': 'o',
        r'[$5]': 's',
        r'[7+]': 't',
        r'[2]': 'z'
    }

    for pattern, repl in patterns.items():
        text = re.sub(pattern, repl, text)

    return text

In [18]:
categories = {
        "negative": [
            "angry", "rage", "pouting", "face_with_symbols_on_mouth",
            "middle_finger", "vomit", "nauseated", "poop",
             "coffin", "bomb"
        ],
        "sarcastic": [
            "joy", "rofl", "laugh", "rolling_on_the_floor_laughing",
            "face_with_tears_of_joy", "smirk", "unamused"
        ],
        "threat": [
            "knife", "dagger", "gun", "pistol", "bomb", "fire",
            "skull", "skull_and_crossbones"
        ],
        "positive": [
            "smile", "grin", "heart", "thumbs_up", "clap",
            "sparkles", "star", "fire"
        ],
        "sad": [
            "cry", "sob", "disappointed", "broken_heart",
            "weary", "tired", "pensive"
        ]
    }



In [19]:
def emoji_sentiment_multi(text):
  scores = {
       "negative": 0,
        "sarcastic": 0,
        "threat": 0,
        "positive": 0,
        "sad": 0
    }
  if not isinstance(text,str):
      return scores

  text = text.lower()



  for label, keywords in categories.items():
        for word in keywords:
            if word in text:
                scores[label] += 1

    # convert to binary (one-hot style)
  return {label: int(score > 0) for label, score in scores.items()}


In [20]:

bad_words = {
    "insult_word": [
        "idiot", "stupid", "dumb", "moron", "loser", "fool"
    ],

    "aggression_word": [
        "hate", "shut up", "go away"
    ],

    "threat_word": [
        "kill", "die", "destroy", "hurt"
    ],

    "profanity_word": [
        "fuck", "shit", "bitch", "asshole", "bastard",
        "damn", "fucker", "bullshit","bastard","ass","dumbass",""
    ]
}


In [21]:


def bad_word_features(text):
    text = text.lower()
    features = {}

    for category, words in bad_words.items():
        count = 0
        for word in words:
            count += len(re.findall(rf"\b{re.escape(word)}\b", text))
        features[category] = count


    total = sum(features.values())
    features["total_bad_words"] = total
    features["bad_word_ratio"] = total/len(text) if len(text)>0 else 0

    features["has_bad_words"] = 1 if total > 0 else 0

    return features

In [22]:
def repetition_features(text):
    repeats = re.findall(r'(.)\1{2,}', text)

    return {
        "repetition_count": len(repeats),
        "has_repetition": int(len(repeats) > 0)
    }

In [23]:
def preprocess(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # clean spam
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)

    #  URLs & emails
    text = re.sub(r'https?://\S+|www\.\S+', ' url ', text)
    text = re.sub(r'\S+@\S+', ' email ', text)

    # numbers
    text = re.sub(r'\d+', ' NUM ', text)

    #  leetspeak normalization
    text = normalize_leetspeak(text)

    # emoji conversion
    text = convert_emoji(text)


    #  clean punctuation
    text = re.sub(r"[^\w\s!?\.]", " ", text)

    #  spaCy processing
    doc = nlp(text)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_space
    ]

    return " ".join(tokens)

train data

In [24]:
train_df.shape

(159571, 7)

In [25]:
m,n=train_df.shape

In [26]:
m

159571

test data

In [27]:
processed_batches = []

batch_size = 100

for i in range(0, train_df.shape[0], batch_size):

    batch = train_df.iloc[i:i+batch_size].copy()



    rep_features = batch["comment_text"].apply(repetition_features)
    batch["comment_text"] = batch["comment_text"].apply(preprocess)
    batch["char_len"] = batch["comment_text"].str.len()
    batch["word_len"] = batch["comment_text"].str.split().str.len()

    bad_features = batch["comment_text"].apply(bad_word_features)
    bad_df = pd.DataFrame(bad_features.tolist())
    rep_df = pd.DataFrame(rep_features.tolist())


    batch["has_emoji"] = batch["comment_text"].apply(has_emoji)
    emoji_sentiment=batch["comment_text"].apply(emoji_sentiment_multi)
    emoji_sentiment_df=pd.DataFrame(emoji_sentiment.to_list())

    batch = pd.concat([
        batch.reset_index(drop=True),
        bad_df.reset_index(drop=True),
        rep_df.reset_index(drop=True),
        emoji_sentiment_df.reset_index(drop=True)
    ], axis=1)

    processed_batches.append(batch)
    print("finished processing instances",i)

# combine all batches
final_train_df = pd.concat(processed_batches, axis=0)



finished processing instances 0
finished processing instances 100
finished processing instances 200
finished processing instances 300
finished processing instances 400
finished processing instances 500
finished processing instances 600
finished processing instances 700
finished processing instances 800
finished processing instances 900
finished processing instances 1000
finished processing instances 1100
finished processing instances 1200
finished processing instances 1300
finished processing instances 1400
finished processing instances 1500
finished processing instances 1600
finished processing instances 1700
finished processing instances 1800
finished processing instances 1900
finished processing instances 2000
finished processing instances 2100
finished processing instances 2200
finished processing instances 2300
finished processing instances 2400
finished processing instances 2500
finished processing instances 2600
finished processing instances 2700
finished processing instances 28

In [ ]:
print("train_df:", train_df.shape)
print("processed_batches:", len(processed_batches))
print("final_df:", final_train_df.shape)

In [30]:
processed_test_batches = []

batch_size = 100

for i in range(0, test_df.shape[0], batch_size):

    batch = test_df.iloc[i:i+batch_size].copy()



    rep_features = batch["comment_text"].apply(repetition_features)
    batch["comment_text"] = batch["comment_text"].apply(preprocess)
    batch["char_len"] = batch["comment_text"].str.len()
    batch["word_len"] = batch["comment_text"].str.split().str.len()

    bad_features = batch["comment_text"].apply(bad_word_features)
    bad_df = pd.DataFrame(bad_features.tolist())
    rep_df = pd.DataFrame(rep_features.tolist())


    batch["has_emoji"] = batch["comment_text"].apply(has_emoji)
    emoji_sentiment=batch["comment_text"].apply(emoji_sentiment_multi)
    emoji_sentiment_df=pd.DataFrame(emoji_sentiment.to_list())

    batch = pd.concat([
        batch.reset_index(drop=True),
        bad_df.reset_index(drop=True),
        rep_df.reset_index(drop=True),
        emoji_sentiment_df.reset_index(drop=True)
    ], axis=1)

    processed_test_batches.append(batch)
    print("finished processing instances",i)





finished processing instances 0
finished processing instances 100
finished processing instances 200
finished processing instances 300
finished processing instances 400
finished processing instances 500
finished processing instances 600
finished processing instances 700
finished processing instances 800
finished processing instances 900
finished processing instances 1000
finished processing instances 1100
finished processing instances 1200
finished processing instances 1300
finished processing instances 1400
finished processing instances 1500
finished processing instances 1600
finished processing instances 1700
finished processing instances 1800
finished processing instances 1900
finished processing instances 2000
finished processing instances 2100
finished processing instances 2200
finished processing instances 2300
finished processing instances 2400
finished processing instances 2500
finished processing instances 2600
finished processing instances 2700
finished processing instances 28

In [32]:
# final_train_df.to_csv("final_toxic_comments_train.csv")
final_test_df = pd.concat(processed_test_batches, axis=0)
final_test_df.to_csv("final_toxic_comments_test.csv",index=False)